# 03 — Feature Harvest and Ensemble Models

This notebook extends the single tree of notebook `02` in two ways, in the order a careful analyst would take them:

1. **A systematic feature harvest.** Notebook `02` used seven predictors, introduced one merge at a time. Here we screen *every* eligible variable in the MTF survey against an explicit rule, then let the models rank them, the same enumerate-then-reduce discipline used in my previous Financial Inclusion project. The full decision record is in `Feature_Harvest_Plan.md` (repo root).
2. **Ensembles.** We compare the tuned decision tree against a **random forest** and **gradient boosting**, tracking every run with **MLflow**, and judge all three by the business metric: precision among the top 20% of ranked households.

**Target (unchanged):** willingness to pay upfront at the randomly offered price (`E_3`; 29% positive).

## The screening rule

Every candidate variable must pass three tests before entering the matrix:

1. **Knowable before outreach**: Obtainable for a household a vendor has not yet contacted.
2. **Not downstream of the target**: Not recorded as a consequence of the `E_3` willingness question. This is the leakage test.
3. **Plausible mechanism**: A defensible economic, infrastructural or behavioural reason to move willingness to pay.

**The decisive exclusion—leakage in Section E.** Section E holds the target `E_3`, but *also* `E_4`, `E_5`, `E_6`, `E_7` and `E_3b`: the price-bargaining follow-ups and the "why did you refuse" fields. These exist only *because* the outcome already happened. Including any of them would inflate the score and then collapse in deployment, where they do not exist. They are excluded by rule, not by trial and error.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../raw_data/extracted/TO UPLOAD")
N_BASE = 3625  # households with a usable target; every merge must preserve this

def check(df, label):
    assert len(df) == N_BASE, f"ROW COUNT CHANGED at {label}: {len(df)} (expected {N_BASE})"
    print(f"[ok] {label:22} {df.shape}")

print("Ready.")

Ready.


## Target construction

From Section E we retain the household key, the willingness-to-pay response, and the randomly offered price. Households never asked the question are dropped; the "don't know" code (888) is excluded by restricting the target to {0, 1}.

In [2]:
sec_e = pd.read_stata(DATA / "MTF_NG_HH_SEC_E.dta", convert_categoricals=False)

target_df = sec_e[["HH_ID", "E_3", "solar_price"]].copy()
target_df = target_df[target_df["E_3"].isin([0, 1])].rename(columns={"E_3": "willing_to_pay"})

df = target_df.copy()
print("Households with a usable target:", len(df))
print(df["willing_to_pay"].value_counts(normalize=True).round(3))

Households with a usable target: 3625
willing_to_pay
0.0    0.71
1.0    0.29
Name: proportion, dtype: float64


## Assembling the candidate matrix

Two data-engineering cautions govern every merge below:

- **Unit of analysis.** Section A1 is *person-level* (22,597 rows over 3,669 households). Head attributes are taken by filtering to the household head (`A4 == 1`); composition features are aggregated. Merging A1 raw would multiply rows and silently corrupt the join.
- **Long-format fuel table.** `F_FUEL` has one row per fuel type per household and is pivoted to household level before merging.

After every merge we assert the row count is still 3,625.

### Location — urban/rural and state

In [3]:
ident = pd.read_stata(DATA / "MTF_NG_HH_Identification.dta", convert_categoricals=False)
df = df.merge(ident[["HH_ID", "Urbanization", "state_id"]], on="HH_ID", how="left")
check(df, "identification")
df["urban"] = (df["Urbanization"] == 2).astype(int)
df = df.drop(columns=["Urbanization"])

[ok] identification         (3625, 5)


### Household head and composition (Section A1, person-level)

**Education keeps the culturally honest recoding of notebook `02`.** The `A9` codes include Quranic schooling (code 51) — the single most common category in this Northern sample — which is not a rung on the formal-education ladder. It is modelled as two variables: an ordered formal band and a separate Quranic indicator. Integrated Quranic schools (code 52), which teach the national curriculum, are counted as primary. Additionally, I included the head's age, gender, and occupation, plus household size and the number of children.

In [4]:
a1 = pd.read_stata(DATA / "MTF_NG_HH_SEC_A1.dta", convert_categoricals=False)

def formal_band(code):
    if code in (11, 12, 13, 14, 15, 16, 52):   # primary (incl. integrated Quranic)
        return 1
    if code in (21, 22, 23, 24, 25, 26, 27, 28):  # secondary
        return 2
    if code in (31, 32, 33, 34, 41, 42, 43):   # post-secondary
        return 3
    return 0                                    # none, nursery, Quranic-only, adult ed

heads = a1[a1["A4"] == 1][["HH_ID", "A9", "A5", "A3", "A14"]].copy()
heads["head_edu_band"] = heads["A9"].apply(formal_band)
heads["head_quranic"]  = heads["A9"].isin([51, 52]).astype(int)
heads["head_age"]      = heads["A5"].where(heads["A5"] < 100, np.nan)  # 888 = missing code
heads["head_female"]   = (heads["A3"] == 2).astype(int)
heads = heads.rename(columns={"A14": "head_occupation"})

df = df.merge(heads[["HH_ID", "head_edu_band", "head_quranic", "head_age",
                     "head_female", "head_occupation"]], on="HH_ID", how="left")
check(df, "head attributes")

df = df.merge(a1.groupby("HH_ID").size().reset_index(name="hh_size"), on="HH_ID", how="left")
check(df, "household size")

kids = a1[a1["A5"] < 15].groupby("HH_ID").size().reset_index(name="n_children")
df = df.merge(kids, on="HH_ID", how="left")
df["n_children"] = df["n_children"].fillna(0)
check(df, "children")

[ok] head attributes        (3625, 10)
[ok] household size         (3625, 11)
[ok] children               (3625, 12)


### Grid connection and Section B (financial inclusion + dwelling quality)

In [5]:
# Grid connection from Section E (a pre-outreach attribute, not downstream of E_3)
df = df.merge(sec_e[["HH_ID", "E1"]].rename(columns={"E1": "grid_connected"}), on="HH_ID", how="left")
check(df, "grid")

sec_b = pd.read_stata(DATA / "MTF_NG_HH_SEC_B.dta", convert_categoricals=False)
sec_b = sec_b.rename(columns={
    "B16": "bank_account", "B21": "mobile_money", "B18": "informal_savings",
    "B7": "own_dwelling", "B9": "n_rooms", "B5": "dwelling_type",
    "B10": "wall_material", "B11": "roof_material", "B12": "floor_material",
    "B14": "water_source", "B4": "years_in_dwelling"})
sec_b["loan_access_breadth"] = sec_b[[f"B20__{i}" for i in range(1, 14)]].sum(axis=1)

bcols = ["HH_ID", "bank_account", "mobile_money", "informal_savings", "own_dwelling",
         "n_rooms", "dwelling_type", "wall_material", "roof_material", "floor_material",
         "water_source", "years_in_dwelling", "loan_access_breadth"]
df = df.merge(sec_b[bcols], on="HH_ID", how="left")
check(df, "section B")

[ok] grid                   (3625, 13)
[ok] section B              (3625, 25)


### Section F: revealing lighting demand

The hypothesis: a household already buying fuel-based light has revealed demand for illumination that solar can capture. We capture both the *use* of fuel lighting and the *amount spent* (the fuel table pivoted to household total). Whether this signal actually holds is left to the model to decide.

In [6]:
sec_f = pd.read_stata(DATA / "MTF_NG_HH_SEC_F.dta", convert_categoricals=False)
sec_f["fuel_light_count"] = (sec_f[["F3__1", "F3__2", "F3__3", "F3__4"]] == 1).sum(axis=1)
sec_f["uses_fuel_light"]  = (sec_f["fuel_light_count"] > 0).astype(int)
df = df.merge(sec_f[["HH_ID", "fuel_light_count", "uses_fuel_light"]], on="HH_ID", how="left")
check(df, "section F (use)")

fuel = pd.read_stata(DATA / "MTF_HH_SEC_F_FUEL.dta", convert_categoricals=False)
spend = fuel.groupby("HH_ID")["F20"].sum().reset_index(name="fuel_spend")  # long -> household
df = df.merge(spend, on="HH_ID", how="left")
df["fuel_spend"] = df["fuel_spend"].fillna(0)
check(df, "fuel spend")

[ok] section F (use)        (3625, 27)
[ok] fuel spend             (3625, 28)


### Section N: asset ownership

Aggregate appliance ownership is a robust wealth proxy. I kept the total asset count plus a few individually meaningful flags (smartphone, TV, radio, fan), and let permutation importance later reveal whether the individual flags add anything beyond the count.

In [7]:
sec_n = pd.read_stata(DATA / "MTF_NG_HH_SEC_N_LIST.dta", convert_categoricals=False)
acols = [f"N_electrical__{i}" for i in range(1, 31)]
sec_n["asset_count"]    = (sec_n[acols] > 0).sum(axis=1)
sec_n["has_smartphone"] = (sec_n["N_electrical__25"] > 0).astype(int)
sec_n["has_tv"]  = ((sec_n["N_electrical__27"] > 0) | (sec_n["N_electrical__28"] > 0) | (sec_n["N_electrical__29"] > 0)).astype(int)
sec_n["has_radio"] = (sec_n["N_electrical__6"] > 0).astype(int)
sec_n["has_fan"]   = (sec_n["N_electrical__8"] > 0).astype(int)
df = df.merge(sec_n[["HH_ID", "asset_count", "has_smartphone", "has_tv", "has_radio", "has_fan"]],
              on="HH_ID", how="left")
check(df, "section N")

[ok] section N              (3625, 33)


### Section L: total consumption (income proxy)

Direct income is not measured — standard for surveys of largely informal economies, where consumption expenditure is the accepted proxy. I sum purchased, produced and gifted consumption across all items to a household total.

In [8]:
sec_l = pd.read_stata(DATA / "MTF_NG_HH_SEC_L_CONSUMPTION.dta", convert_categoricals=False)
for col in ["L_consumption_purchased", "L_consumption_produced", "L_consumption_gift"]:
    sec_l[col] = pd.to_numeric(sec_l[col], errors="coerce").fillna(0)
sec_l["total"] = sec_l[["L_consumption_purchased", "L_consumption_produced", "L_consumption_gift"]].sum(axis=1)
cons = sec_l.groupby("HH_ID")["total"].sum().reset_index(name="total_consumption")
df = df.merge(cons, on="HH_ID", how="left")
check(df, "consumption")

[ok] consumption            (3625, 34)


### The assembled matrix

In [9]:
features = [c for c in df.columns if c not in ("HH_ID", "willing_to_pay")]
X = df[features].fillna(df[features].median())
y = df["willing_to_pay"].astype(int)

print(f"Design matrix: {X.shape[0]} households x {X.shape[1]} features")
print("Missingness after median fill:", int(X.isna().sum().sum()))
print("\nFeatures:")
print(features)

Design matrix: 3625 households x 32 features
Missingness after median fill: 0

Features:
['solar_price', 'state_id', 'urban', 'head_edu_band', 'head_quranic', 'head_age', 'head_female', 'head_occupation', 'hh_size', 'n_children', 'grid_connected', 'bank_account', 'mobile_money', 'informal_savings', 'own_dwelling', 'n_rooms', 'dwelling_type', 'wall_material', 'roof_material', 'floor_material', 'water_source', 'years_in_dwelling', 'loan_access_breadth', 'fuel_light_count', 'uses_fuel_light', 'fuel_spend', 'asset_count', 'has_smartphone', 'has_tv', 'has_radio', 'has_fan', 'total_consumption']


## Modelling: one tree vs. an ensemble of many

I compared three models, each tuned by 5-fold `GridSearchCV` on ROC-AUC:

- **Decision tree**: The notebook `02` baseline, re-tuned on the wider feature set.
- **Random forest**: Many decorrelated trees voting; reduces *variance*. Decorrelation comes from bootstrap sampling *and* random feature subsetting at each split.
- **Gradient boosting**: Trees built *sequentially*, each correcting the last; reduces *bias*.

Every run is logged to **MLflow**. This parameter sweep across three model families is exactly the experiment-tracking problem MLflow exists for. The evaluation metric is **precision at the top 20%** of ranked households, compared against the random-outreach base rate.

> **A note on where tracking belongs.** In a production workflow, experiment tracking lives in a training *script* or pipeline, not a notebook — the notebook is for exploration and explanation. MLflow is used here, in the notebook, so the tracking sits alongside the reasoning; in deployment it would move into the batch-scoring service. MLflow is a declared dependency in `requirements.txt`, and the `mlruns/` store and `mlflow.db` are git-ignored (generated artifacts, like `__pycache__`). A screenshot of the run comparison is committed at the repo root (`mlflow_run_comparison.png`) as portable evidence for anyone who reads the repo without running it.

In [10]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

def precision_at_top(frac, scores, y_true):
    k = int(len(scores) * frac)
    top = np.argsort(scores)[::-1][:k]
    return float(y_true.iloc[top].mean())

base_rate = float(y_test.mean())
cv = StratifiedKFold(5, shuffle=True, random_state=42)
print(f"Random-outreach base rate (test set): {base_rate:.3f}")

Random-outreach base rate (test set): 0.290


In [11]:
import mlflow

# SQLite backend (not the deprecated file store): portable, single-file, and the
# path MLflow 3.x recommends. The mlflow.db file is git-ignored.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("solar_ensembles")
print("MLflow tracking ->", mlflow.get_tracking_uri())

MLflow disabled: No module named 'mlflow'


In [12]:
configs = {
    "decision_tree": (DecisionTreeClassifier(random_state=42),
        {"max_depth": [4, 5, 6, 7], "min_samples_leaf": [30, 50, 80]}),
    "random_forest": (RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators": [300], "max_depth": [10, None],
         "min_samples_leaf": [5, 10, 20], "max_features": ["sqrt"]}),
    "gradient_boosting": (GradientBoostingClassifier(random_state=42),
        {"n_estimators": [150, 250], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}),
}

results, fitted = [], {}
for name, (est, grid) in configs.items():
    gs = GridSearchCV(est, grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)
    model = gs.best_estimator_
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    p20 = precision_at_top(0.20, proba, y_test)
    lift = p20 / base_rate
    fitted[name] = model
    results.append({"model": name, "test_roc_auc": round(auc, 3),
                    "precision_top20": round(p20, 3), "lift_vs_random": round(lift, 2)})
    with mlflow.start_run(run_name=name):
        mlflow.log_params(gs.best_params_)
        mlflow.log_metric("cv_roc_auc", float(gs.best_score_))
        mlflow.log_metric("test_roc_auc", float(auc))
        mlflow.log_metric("precision_at_top20", float(p20))
        mlflow.log_metric("lift_vs_random", float(lift))

pd.DataFrame(results).set_index("model")

,test_roc_auc,precision_top20,lift_vs_random
model,,,
decision_tree,0.733,0.519,1.79
random_forest,0.799,0.608,2.10
gradient_boosting,0.796,0.635,2.19


## Reading the result

The ensembles beat the single tree: a single tree is low-bias but high-variance: it overreacts to noise in its particular training sample. Averaging many *decorrelated* trees cancels that random error (random forest), while boosting attacks the residual bias by correcting errors in sequence. Gradient boosting gives the strongest top-20% precision, which is the number the client's outreach budget actually feels.

Note a subtlety worth internalising: `GridSearchCV` optimises **ROC-AUC**, not precision-at-top-20%. The two usually move together, but not always. A configuration can win on AUC yet rank the very top of the list slightly worse. When the deployment decision is purely "which fifth do we contact," it is legitimate to tune directly on the business metric; we keep AUC here for comparability with notebook `02`.

## Which features actually earned their place

Feature importances on wide, correlated feature sets can mislead, so we use **permutation importance** on the held-out test set: how much does test ROC-AUC fall when each feature is shuffled? This measures out-of-sample contribution, not in-sample splitting frequency.

In [ ]:
from sklearn.inspection import permutation_importance

best_name = max(results, key=lambda r: r["lift_vs_random"])["model"]
best_model = fitted[best_name]
pi = permutation_importance(best_model, X_test, y_test,
                            scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=-1)

imp = (pd.DataFrame({"feature": features, "importance": pi.importances_mean})
         .sort_values("importance", ascending=False).reset_index(drop=True))
print(f"Permutation importance — {best_name}")
imp

### What the harvest taught us

Three honest lessons from the importance ranking:

1. **The harvest paid off — but not where I expected.** The features that lift performance beyond notebook `02` are the **asset count**, **number of rooms**, **total consumption**, and **state** (geography) — wealth and location proxies. The offered price still dominates, as the experiment guarantees.

2. **My headline hypothesis failed, and the data gets the last word.** I expected *fuel-lighting spend* to be a strong signal — the intuitive "revealed demand" story. In this sample it is near-zero: very few households report fuel-based lighting purchases, and the model finds no signal there. A documented wrong prediction is more trustworthy than a quiet one — the same lesson the urbanization bet taught in notebook `02`.

3. **Collinearity, made visible.** Individually, `has_fan` / `has_tv` / `has_smartphone` looked correlated with the target, yet their *permutation* importance is near-zero — because `asset_count` already absorbs them. This is why permutation importance on the ensemble, not univariate correlation, is the honest basis for pruning.

**Pruning decision:** the fuel variables, individual appliance flags, dwelling materials, `loan_access_breadth`, and `hh_size` contribute effectively nothing and can be dropped in a lean production model with no loss — keeping only price, the asset/consumption/rooms wealth proxies, education (incl. the Quranic indicator), grid connection, and state.

## Conclusion and next steps

Ensembling plus a disciplined feature harvest raises top-20% precision meaningfully above the single-tree baseline, with gradient boosting the strongest model. The gain comes from wealth and location signals the harvest surfaced — and, just as informatively, *not* from the fuel-demand story that intuition favoured.

**For the client**, the operational message is unchanged in shape but sharper in content: rank households by the model, contact the top fifth, and expect roughly double the conversion of untargeted outreach — with asset ownership, dwelling size, consumption, and state as the levers behind the offered price.

**Next in the roadmap:**
- Tune directly on precision-at-top-20% and add fairness diagnostics across gender and urban/rural segments.
- Deploy the winning model as a **batch-scoring service** (household CSV in → ranked contact list out), containerised with **Docker**, with MLflow carried through from this notebook.
- A two-page client-style targeting brief built on these results.